## Expérimentation de modèles

In [4]:
import joblib
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, median_absolute_error, r2_score, get_scorer_names

In [15]:
train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_parquet("../data/test_data.parquet")

X_train = train_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_train = train_data["LoyerMensuel_Log1"]

print(X_train)

     Chambres  Superficie_m2  DistanceRoute_m  AgeMaison  Salon_Bin  \
0         1.0           46.0            154.0       20.0          1   
1         4.0          217.0            211.0       23.0          1   
2         4.0          181.0             64.0       24.0          1   
3         6.0          253.0            255.0        2.0          1   
4         3.0          162.0             29.0       34.0          1   
..        ...            ...              ...        ...        ...   
377       4.0          183.0            213.0       14.0          1   
378       6.0          171.0            250.0       34.0          1   
379       5.0          223.0            293.0       16.0          1   
380       6.0          287.0             24.0       30.0          1   
381       6.0          272.0            235.0       31.0          1   

     SalleDeBainInterieure_Bin  Parking_Bin  Meuble_Bin  Jardin_Bin  \
0                            1            0           0           0   
1    

### Transformation de données de test

In [16]:
# Les variables numériques
feature_cols = ["AgeMaison", "Quartier_Target", "Indicateur_Confort", "Chambres_par_Superficie", "LoyerMensuel_Log1"]

for col in ['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin']:
    test_data[col + "_Bin"] = test_data[col].map({"Oui": 1, "Non": 0}).fillna(0).astype(int)

neighbourhood_encoder = joblib.load("neighbourhood_encoder.joblib") 
test_data["Quartier_Target"] = neighbourhood_encoder.transform(test_data[["Quartier"]])[:, 0]

test_data["LoyerMensuel_Log1"] = np.log1p(test_data["LoyerMensuel_BIF"])

# Création de variables utiles pour le test
# cols_confort = ['Salon_Bin', 'SalleDeBainInterieure_Bin', 'Parking_Bin', 'Meuble_Bin', 'Jardin_Bin']
# test_data['Indicateur_Confort'] = test_data[cols_confort].sum(axis=1)

# test_data['Chambres_par_Superficie'] = test_data['Chambres'] / (test_data['Superficie_m2'] + 0.1)

# Séparation de X_test et y_test
X_test = test_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_test = test_data["LoyerMensuel_Log1"]

X_test

,Chambres,Superficie_m2,DistanceRoute_m,AgeMaison,Salon_Bin,SalleDeBainInterieure_Bin,Parking_Bin,Meuble_Bin,Jardin_Bin,Quartier_Target
0,1.0,46.0,154.0,20.0,1,1,0,0,0,0.010428
1,4.0,217.0,211.0,23.0,1,1,0,0,0,0.002128
2,4.0,181.0,64.0,24.0,1,1,1,0,0,0.002182
3,6.0,253.0,255.0,2.0,1,1,0,0,1,0.001636
4,3.0,162.0,29.0,34.0,1,1,1,1,0,0.001636
...,...,...,...,...,...,...,...,...,...,...
377,4.0,183.0,213.0,14.0,1,1,0,0,1,0.002163
378,6.0,171.0,250.0,34.0,1,1,0,0,0,0.001636
379,5.0,223.0,293.0,16.0,1,1,0,0,1,0.002061
380,6.0,287.0,24.0,30.0,1,1,1,0,1,0.001636


In [17]:
def formated_time(second):
    m, s = divmod(second, 60)

    if m > 0:
         f"{int(m)} min {s:.4f} secondes"
    else:
        return f"{s:.4f} secondes"

scoring = ["neg_mean_squared_error", "neg_median_absolute_error", "neg_root_mean_squared_error", "r2"]

### Modèle de Régression Linéaire

In [18]:
linear_model = LinearRegression()

linear_scores = cross_validate(linear_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {linear_scores["fit_time"]}")
print()
print(f"Score Time : {linear_scores["score_time"]}")
print()
print(f"Test Neg Median Abosulte Error : {linear_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg Mean Squared Error : {linear_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test R2 : {linear_scores["test_r2"]}")

Fit Time : [0.00948501 0.00678229 0.00682878 0.00601554 0.00900388]

Score Time : [0.00864553 0.00878739 0.00868893 0.00765872 0.01234651]

Test Neg Median Abosulte Error : [-0.29193164 -0.33508889 -0.31824578 -0.24118789 -0.3432502 ]

Test Neg Mean Squared Error : [-0.28048846 -0.19495255 -0.19231695 -0.14773481 -0.16391579]

Test R2 : [0.38511628 0.60438951 0.39594465 0.5010853  0.51968459]


### Modèle Lasso

In [19]:
lasso_model = Lasso(alpha=0.2)

lasso_scores = cross_validate(lasso_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {lasso_scores["fit_time"]}")
print()
print(f"Score Time : {lasso_scores["score_time"]}")
print()
print(f"Test Neg MSE : {lasso_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {lasso_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {lasso_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {lasso_scores["test_r2"]}")

Fit Time : [0.05719757 0.00445461 0.01366925 0.01007986 0.00405169]

Score Time : [0.01338816 0.00993967 0.00800538 0.01135755 0.00469184]

Test Neg MSE : [-0.33449344 -0.29258357 -0.24175764 -0.1838751  -0.21524436]

Test Neg  MAE : [-0.33310388 -0.40793512 -0.34381955 -0.28912704 -0.35920642]

Test Neg SMSE : [-0.57835408 -0.54090994 -0.49168856 -0.4288066  -0.46394435]

Test R2 : [0.26672713 0.40627025 0.24065459 0.37903606 0.36927869]


### Ridget Model

In [20]:
ridge_model = Ridge(alpha=0.1)
ridge_scores = cross_validate(ridge_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {ridge_scores["fit_time"]}")
print()
print(f"Score Time : {ridge_scores["score_time"]}")
print()
print(f"Test Neg MSE : {ridge_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {ridge_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {ridge_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {ridge_scores["test_r2"]}")

Fit Time : [0.00589108 0.0053885  0.00710797 0.00705528 0.0083909 ]

Score Time : [0.00754166 0.00596809 0.00883412 0.00822568 0.01553917]

Test Neg MSE : [-0.29862461 -0.2346449  -0.2073119  -0.18691915 -0.17880967]

Test Neg  MAE : [-0.31292038 -0.37353231 -0.35652329 -0.31659235 -0.35688442]

Test Neg SMSE : [-0.54646556 -0.48440159 -0.45531516 -0.43234148 -0.42285892]

Test R2 : [0.34535839 0.52384318 0.34884646 0.368756   0.4760417 ]


### Arbres de décision

In [24]:
tree_model = DecisionTreeRegressor()
tree_scores = cross_validate(tree_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {tree_scores["fit_time"]}")
print()
print(f"Score Time : {tree_scores["score_time"]}")
print()
print(f"Test Neg MSE : {tree_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {tree_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {tree_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {tree_scores["test_r2"]}")

Fit Time : [0.00972414 0.00649405 0.00480819 0.0104053  0.01060939]

Score Time : [0.00696969 0.00747848 0.00771594 0.01710629 0.01621652]

Test Neg MSE : [-0.29921674 -0.22353259 -0.246832   -0.2296483  -0.24846851]

Test Neg  MAE : [-0.26218046 -0.24611896 -0.29804017 -0.2939702  -0.31982859]

Test Neg SMSE : [-0.54700708 -0.47279233 -0.4968219  -0.47921634 -0.49846616]

Test R2 : [0.34406033 0.54639301 0.22471633 0.22445557 0.27192337]


### Modèle de Fôrets Aléatoire

In [22]:
random_model = RandomForestRegressor()

random_scores = cross_validate(random_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {random_scores["fit_time"]}")
print()
print(f"Score Time : {random_scores["score_time"]}")
print()
print(f"Test Neg MSE : {random_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {random_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {random_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {random_scores["test_r2"]}")

Fit Time : [0.26220322 0.21902108 0.21544957 0.21207881 0.21508479]

Score Time : [0.01881599 0.01862121 0.01831222 0.01763821 0.01878071]

Test Neg MSE : [-0.20597569 -0.14762321 -0.10751961 -0.10399237 -0.11694806]

Test Neg  MAE : [-0.21764308 -0.26741775 -0.20359778 -0.21095196 -0.25821945]

Test Neg SMSE : [-0.45384545 -0.38421766 -0.32790183 -0.32247848 -0.3419767 ]

Test R2 : [0.54846234 0.70043331 0.66228771 0.64880773 0.65731211]


### Modèle de Gradient Boosting

In [26]:
boosting_model = GradientBoostingRegressor(random_state=0)

boosting_scores = cross_validate(boosting_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {boosting_scores["fit_time"]}")
print()
print(f"Score Time : {boosting_scores["score_time"]}")
print()
print(f"Test Neg MSE : {boosting_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test Neg  MAE : {boosting_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg SMSE : {boosting_scores["test_neg_root_mean_squared_error"]}")
print()
print(f"Test R2 : {boosting_scores["test_r2"]}")

Fit Time : [0.15012741 0.09846258 0.10257864 0.09412503 0.09392214]

Score Time : [0.00474644 0.00499415 0.00699067 0.00537753 0.00554609]

Test Neg MSE : [-0.18749778 -0.07400438 -0.05695738 -0.05778474 -0.06539959]

Test Neg  MAE : [-0.1438035  -0.16189354 -0.15416844 -0.17510174 -0.16474067]

Test Neg SMSE : [-0.43301013 -0.27203746 -0.23865746 -0.24038456 -0.25573343]

Test R2 : [0.58896943 0.84982547 0.82110047 0.80485537 0.8083624 ]
